**Import the required libraries:**

In [73]:
%pip install langchain langchain-community langchain-openai langchain-chroma chromadb langgraph ddgs python-dotenv scikit-learn

  Using cached websockets-16.1.1-cp313-cp313-win_amd64.whl.metadata (7.0 kB)
Using cached websockets-16.1.1-cp313-cp313-win_amd64.whl (180 kB)
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   -- ------------------------------------- 0.3/4.0 MB ? eta -:--:--
   -- ------------------------------------- 0.3/4.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.5/4.0 MB 659.7 kB/s eta 0:00:06
   ----- ---------------------------------- 0.5/4.0 MB 659.7 kB/s eta 0:00:06
   ------- -------------------------------- 0.8/4.0 MB 612.4 kB/s eta 0:00:06
   ------- -------------------------------- 0.8/4.0 MB 612.4 kB/s eta 0:00:06
   ------- -------------------------------- 0.8/4.0 MB 612.4 kB/s eta 0:00:06
   ---------- ----------------------------- 1.0/4.0 MB 601.2 kB/s eta 0:00:05
   ------------- -------------------------- 1.3/4.0 MB 624.7 kB/s eta 0:00:05
   ------------- ------------------


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from langchain_chroma import Chroma
import os
import chromadb
import pathlib
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv

Loading the Documents 

In [67]:
# Source 1 - the PDF. PyPDFLoader returns one document per page.

pdf_docs_fixed = PyMuPDFLoader("Lagos_Education_RAG_Knowledge_Base.pdf").load()

# Source 2 - the text file. Plain Python can read this on its own.
with open("Lagos_Resource_Gap_Ranking.txt", "r", encoding="utf-8") as f:
    txt_text = f.read()

print("Pages loaded from the PDF :", len(pdf_docs_fixed))
print("Characters read from TXT  :", len(txt_text))

print()
print("First 1000 characters of the PDF:")
print(pdf_docs_fixed[0].page_content[:1000])

print()
print("First 300 characters of the TXT:")
print(txt_text[:300])


Pages loaded from the PDF : 33
Characters read from TXT  : 1464

First 1000 characters of the PDF:
Page 1
Lagos LGA Education Resources: RAG
Knowledge Base
Education Resource-Gap Intelligence Tool · 20 Lagos LGAs · 2016/17–2019/20 and 2024/25
About This Document
This document describes the education resources of public and private schools in the 20
Local Government Areas (LGAs) of Lagos State, Nigeria. It is the knowledge base for the
Education Resource-Gap Intelligence Tool. The analysis is at LGA level, not individual
school level.
The data comes from two official sources. The Lagos Bureau of Statistics report 'Spotlight on
Lagos Statistics 2021' provides public school enrolment (by gender) and student-teacher
ratios for Primary, JSS and SSS from 2016/17 to 2019/20. The DNEMIS Annual School
Census 2024/25 (Federal Ministry of Education) provides learners, student-teacher, student-
classroom and student-toilet ratios for public schools by level and for private schools as an
LGA total.

**Splitting the documents into chunks**
A whole PDF page or a whole text file is usually too long to embed as a single
vector. Long text also makes retrieval less precise, because one vector has to
represent many different ideas at once.
So we cut each document into smaller overlapping **chunks**:

- `chunk_size=1200` - each chunk is about 1200 characters.
- `chunk_overlap=50` - each chunk repeats the last 50 characters of the one
  before it, so a sentence is not cut in half and loses its meaning.

In [42]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=50
)

# The PDF pages are already documents, so we split the documents
pdf_chunks = splitter.split_documents(pdf_docs_fixed)

# The TXT is one long plain string, so we create documents from it
txt_chunks = splitter.create_documents([txt_text])

print("Chunks from the PDF :", len(pdf_chunks))
print("Chunks from the TXT :", len(txt_chunks))

# CHOOSE WHICH SOURCE TO EMBED
chosen_chunks = pdf_chunks + txt_chunks            


# Chroma only needs the plain text of each chunk
documents = [chunk.page_content for chunk in chosen_chunks]

print()
print("Documents ready to embed:", len(documents))

Chunks from the PDF : 131
Chunks from the TXT : 2

Documents ready to embed: 133


In [41]:
# Comfirm the of number documents to embed 
print(f"Total Documents to be embedded {len(documents)}\n")

Total Documents to be embedded 133



Loading the API key from our `.env` file.

In [23]:
load_dotenv()

# ---------------------------------------------------------------
# PROVIDER CONFIG
# The API keys are secret, so they live in the .env file.
# Everything else is written out here so you can see it.
#
# NOTE: the chat model and the embedding model run on TWO DIFFERENT
# servers. Each one needs its own base URL and its own key.
# Do not mix them up.
# ---------------------------------------------------------------

# --- Chat model ---
api_key         = os.getenv("OPENAI_API_KEY")
# base_url        = "https://gemma-4.publicaai.com/v1"
# chat_model_name = "llama-3.3-70b-versatile"
base_url= "https://api.groq.com/openai/v1"
chat_model_name= "openai/gpt-oss-20b"

# --- Embedding model (different server, different key) ---
embedding_api_key    = os.getenv("EMBEDDING_API_KEY")
embedding_base_url   = "https://qwen-embed.publicaai.com/v1"
embedding_model_name = "Qwen/Qwen3-Embedding-0.6B"

print("Chat base URL     :", base_url)
print("Chat model        :", chat_model_name)
print("Embed base URL    :", embedding_base_url)
print("Embedding model   :", embedding_model_name)

Chat base URL     : https://api.groq.com/openai/v1
Chat model        : openai/gpt-oss-20b
Embed base URL    : https://qwen-embed.publicaai.com/v1
Embedding model   : Qwen/Qwen3-Embedding-0.6B


In [24]:
# Check whether Python is loading both keys

print("Chat API key loaded:", bool(api_key))
print("Embedding API key loaded:", bool(embedding_api_key))

Chat API key loaded: True
Embedding API key loaded: True


In [25]:
# Setting Up ChromaDB for embeddings Storage
# chroma_path: Creates a folder named chroma_store to store the vector embeddings.

chroma_path = "chroma_store"


**We are using the `embedding_model_name` above** to convert documents into vector embeddings.


In [26]:
# Initialize the embedding model
embeddings = OpenAIEmbeddings(
    model=embedding_model_name,
    api_key=embedding_api_key,
    base_url=embedding_base_url

)

- Creates a **vector database (Chroma)** for the documents.
- Adds the **embedded texts** to the database.

In [27]:
# Initialize ChromaDB
# lerkb = lagos education rag knowledge base
lerkbvdb = Chroma(
    collection_name="lerkb",
    embedding_function=embeddings,
    persist_directory=chroma_path
)


#### The Retriever
Retrieving Documents for a Query

**Generating Responses Using openai/gpt-oss-20b  model**
- **`temperature= 0` ensures deterministic outputs.**

In [32]:
#initialize the chatmodel
chatmodel = ChatOpenAI(
    api_key=api_key,
    base_url=base_url,
    model=chat_model_name,
    temperature=0
)

### **Prompting the LLM**

In [ ]:
#using from template method

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """You are an education resource-gap assistant for Lagos State, Nigeria.

            You will be provided with context retrieved from the
            Lagos LGA Education Resources knowledge base.

            Use the provided context to answer the user's question
            accurately and clearly.

            Focus on education resource gaps across the 20 LGAs
            of Lagos State, including:
            - Learner-teacher ratios
            - Learner-classroom ratios
            - Learner-toilet ratios
            - LGAs with the greatest and smallest resource gaps
            - National resource standards
            - Historical education enrolment and resource data

            Base your answers strictly on the provided context.
            Do not invent statistics, rankings, or facts.
            If the context does not contain enough information,
            clearly state that the information is not available
            in the knowledge base.

            Distinguish between historical data from 2016/17–2019/20
            and the 2024/25 school census data.

            Explain technical terms in simple language.
            When giving statistics or rankings, include the relevant
            figures and identify the year or period where available.

            Keep responses relevant, concise, and well-structured.
            """
        ),
        ("human", "Context: {context}\n\nQuestion: {question}")
    ]
)

question = (
    "Which five LGAs have the greatest overall education "
    "resource gaps in Lagos State, and why?"
)

get_doc = lerkbvdb.as_retriever(
    search_kwargs={"k": 5}
).invoke(question)

context = "\n\n".join(
    doc.page_content for doc in get_doc
)

input_data = {
    "context": context,
    "question": question
}

chain = prompt | chatmodel | StrOutputParser()

result = chain.invoke(input_data)

print(result)

**Answer based on the supplied context**

| LGA | Evidence of resource gap | Why it is a gap |
|-----|---------------------------|-----------------|
| **Lagos Mainland** | • Rank 18 of 20 for classroom shortage (25.5 students per classroom) <br>• Rank 18 of 20 for toilet shortage (25.5 students per toilet) <br>• Overall rank 19 of 20 for resource needs (1 = most lacking) | The LGA is near the bottom of the ranking for both classroom and toilet shortages, and its overall resource‑needs score is the second‑worst in the state. |

**What the context tells us**

- The only LGA for which a ranking of overall resource needs is provided is Lagos Mainland, which is ranked 19 of 20 (the second‑worst).  
- No other LGAs in the supplied excerpts include a comparable overall ranking or a combined assessment of teacher, classroom, and toilet shortages.

**What we cannot say**

- The context does **not** provide overall resource‑gap rankings for the other 19 LGAs, so we cannot identify the five LGAs 

To ensure ranking questions use the ranking TXT, while detailed education questions should use the full knowledge base.
* What this does:
- ranking_retriever searches only the two TXT chunks containing the ranking.
- general_retriever searches only the PDF chunks containing resource ratios, standards, LGA information and historical data.

In [60]:
# Retriever for overall resource-gap rankings
ranking_retriever = lerkbvdb.as_retriever(
    search_kwargs={
        "k": 2,
        "filter": {
            "source": "Lagos_Resource_Gap_Ranking.txt"
        }
    }
)

# Retriever for the general education knowledge base
general_retriever = lerkbvdb.as_retriever(
    search_kwargs={
        "k": 5,
        "filter": {
            "source": "Lagos_Education_RAG_Knowledge_Base.pdf"
        }
    }
)

print("Both retrievers are ready!")

Both retrievers are ready!


In [69]:
# Create a question-routing function
# So that Questions about rankings 
# and the highest or lowest resource-gap scores will use the TXT ranking; 
# other questions will use the PDF.

# Route ranking questions to the ranking TXT and other questions to the education PDF.

def retrieve_context(question):
    ranking_keywords = [
        "ranking",
        "ranked",
        "greatest overall",
        "greatest resource gap",
        "greatest resource gaps",
        "largest resource gap",
        "largest resource gaps",
        "highest resource gap",
        "highest resource gaps",
        "highest resource-gap score",
        "lowest resource gap",
        "lowest resource gaps",
        "smallest resource gap",
        "smallest resource gaps",
        "top five",
        "bottom five",
        "five lgas",
        "which lgas",
        "overall resource gap",
        "overall resource gaps",
        "most lacking overall",
        "worst resource gaps",
        "best-resourced lgas",
        "resource-gap score",
        "resource gap scores",
    ]

    # Normalize the question to make keyword matching case-insensitive.
    question_lower = question.lower()

    # Check whether the question is about the overall LGA ranking.
    is_ranking_question = any(
        keyword in question_lower for keyword in ranking_keywords
    )

    if is_ranking_question:
        # Search the dedicated ranking file rather than the general PDF.
        docs = lerkbvdb.similarity_search(
            "LAGOS STATE EDUCATION RESOURCE-GAP RANKING",
            k=2,
            filter={"source": "Lagos_Resource_Gap_Ranking.txt"}
        )

        # Put the chunk containing the ranking title first when available.
        docs.sort(
            key=lambda doc: (
                0 if "LAGOS STATE EDUCATION RESOURCE-GAP RANKING"
                in doc.page_content.upper() else 1
            )
        )

        source_used = "Ranking TXT"

    else:
        # Use the PDF for general questions about education resources.
        docs = general_retriever.invoke(question)
        source_used = "Education PDF"

    # Combine the retrieved documents into the context for the language model.
    context = "\n\n".join(doc.page_content for doc in docs)

    return context, source_used


In [71]:
def ask_education_rag(question):
    context, source_used = retrieve_context(question)

    input_data = {
        "context": context,
        "question": question
    }

    chain = prompt | chatmodel | StrOutputParser()
    answer = chain.invoke(input_data)

    print("Source used:", source_used)
    print("\nAnswer:\n")
    print(answer)

In [72]:
# Test the complete RAG system using the routing function in Cell 23.
ask_education_rag("Which five LGAs have the greatest overall education resource gaps?")
ask_education_rag("Which LGA has the highest resource-gap score?")
ask_education_rag("Which five LGAs have the smallest resource gaps?")
ask_education_rag("What is the learner-to-teacher ratio standard?")
ask_education_rag("Which environmental initiatives are covered in the education resource data?")



Source used: Ranking TXT

Answer:

The five LGAs with the largest overall education resource gaps (highest resource‑gap scores) are:

| Rank | LGA | Resource‑gap score (out of 100) |
|------|-----|---------------------------------|
| 1 | Badagry | 100.00 |
| 2 | Ajeromi/Ifelodun | 91.09 |
| 3 | Ojo | 85.40 |
| 4 | Ibeju/Lekki | 73.65 |
| 5 | Alimosho | 70.85 |

These scores come directly from the Lagos State Education Resource‑Gap Ranking provided in the context.
Source used: Ranking TXT

Answer:

The LGA with the highest resource‑gap score is **Badagry**, with a score of **100.00 out of 100**.
Source used: Ranking TXT

Answer:

The five LGAs with the **smallest resource gaps** (i.e., the lowest resource‑gap scores) are:

| Rank | LGA | Resource‑gap score (out of 100) |
|------|-----|---------------------------------|
| 20 | Mushin | 8.25 |
| 19 | Lagos Mainland | 14.59 |
| 18 | Oshodi/Isolo | 15.78 |
| 17 | Kosofe | 18.36 |
| 16 | Surulere | 22.49 |

These LGAs have the lowest relativ